# NumPy & Pandas for Software Engineers
## Beyond Data Science: Everyday SE Tasks at Array Speed

> **Audience:** Python engineers who are productive with lists, loops, and comprehensions — but haven't yet reached for NumPy/Pandas outside a data-science context.
>
> This notebook shows **18 concrete, copy-pasteable examples** where vectorised operations replace the kind of boilerplate you write every sprint.  
> Each section has a *"Why bother?"* note and a timing comparison so you can feel the difference.

### What you'll need
```
uv pip install numpy pandas --system
```

### Mental model in one sentence
> A NumPy `ndarray` or a Pandas `Series` is like a SQL column: operations apply to **every element simultaneously**, with no Python loop overhead.

In [ ]:
"""Shared imports and tiny helpers used throughout the notebook."""

from __future__ import annotations

import re
import time
import random
import string
import json
from contextlib import contextmanager
from collections.abc import Callable
from typing import Any

import numpy as np
import pandas as pd

# ── reproducibility ──────────────────────────────────────────────────────────
RNG = np.random.default_rng(42)
random.seed(42)

# ── micro-benchmark helper ────────────────────────────────────────────────────
@contextmanager
def timer(label: str) -> None:
    """Context manager that prints wall-clock elapsed time.

    Args:
        label: Human-readable description of the block being timed.

    Yields:
        None

    Example:
        >>> with timer("my loop"):
        ...     _ = [x**2 for x in range(10_000)]
        my loop: 1.2 ms
    """
    t0 = time.perf_counter()
    yield
    ms = (time.perf_counter() - t0) * 1_000
    print(f"  {label}: {ms:.2f} ms")


print("✓ imports OK — numpy", np.__version__, "| pandas", pd.__version__)

---
## Part 1 — String Operations

The single biggest surprise for SE newcomers: **NumPy and Pandas have a full vectorised string API** (`np.char.*` and `Series.str.*`).  
Everything that `str` can do, you can apply to a million strings without a loop.

### Example 1 — Batch string lengths (and any single-arg callable)
> **Why bother?**  Validating token lengths, truncating payloads, enforcing field-width contracts — all without a loop.

In [ ]:
"""Example 1: vectorised string lengths."""

N = 100_000


def make_random_strings(n: int, max_len: int = 80) -> list[str]:
    """Generate a list of random ASCII strings.

    Args:
        n: Number of strings to generate.
        max_len: Maximum length of each string.

    Returns:
        List of random strings with lengths in [1, max_len].
    """
    return [
        "".join(random.choices(string.ascii_lowercase, k=random.randint(1, max_len)))
        for _ in range(n)
    ]


tokens: list[str] = make_random_strings(N)
arr: np.ndarray = np.array(tokens)  # O(n) copy — do once, reuse many times

# ── the idiom ────────────────────────────────────────────────────────────────
np_len = np.vectorize(len)          # wrap ANY Python callable

with timer("list comprehension"):
    loop_lens = [len(s) for s in tokens]

with timer("np.char.str_len  (built-in)"):
    vec_lens = np.char.str_len(arr) # preferred for len()

with timer("np.vectorize(len)"):
    vec_lens2 = np_len(arr)         # generic pattern for custom funcs

assert list(vec_lens) == loop_lens
print(f"\nSample lengths: {vec_lens[:5]}")
print(f"tokens > 60 chars: {(vec_lens > 60).sum():,}")   # boolean mask, no loop

### Example 2 — Normalise / sanitise a batch of identifiers
> **Why bother?**  Service names, feature flags, config keys arriving from different sources need consistent casing and trimming before they hit a registry or DB.

In [ ]:
"""Example 2: batch identifier normalisation with pandas str accessor."""

raw_keys: list[str] = [
    "  Auth-Service ",
    "PAYMENT_SERVICE",
    "  inventory-service\t",
    "UserService",
    "NOTIFICATION_SVC  ",
    "search-SERVICE",
]

s = pd.Series(raw_keys)

# chain .str methods — each returns a new Series, no loop needed
normalised: pd.Series = (
    s.str.strip()           # remove surrounding whitespace
     .str.lower()           # lowercase
     .str.replace(r"[-_\s]+", "-", regex=True)  # canonicalise separators
)

print("Before → After")
for before, after in zip(s, normalised):
    print(f"  {before!r:30s} → {after!r}")

### Example 3 — Batch regex extraction (structured field parsing)
> **Why bother?**  You receive 50 000 event strings and need to pull out a version tag, a UUID, or an error code from each one. `Series.str.extract` returns a tidy DataFrame in one call.

In [ ]:
"""Example 3: structured field extraction with str.extract."""

import uuid as _uuid


def make_event_lines(n: int) -> list[str]:
    """Generate synthetic event log lines.

    Args:
        n: Number of lines to generate.

    Returns:
        List of strings like ``"deploy service=auth version=2.3.1 id=<uuid>"``.
    """
    services = ["auth", "payment", "inventory", "search", "notification"]
    return [
        f"deploy service={random.choice(services)} "
        f"version={random.randint(1,9)}.{random.randint(0,19)}.{random.randint(0,9)} "
        f"id={_uuid.uuid4()}"
        for _ in range(n)
    ]


events = pd.Series(make_event_lines(10))

# Named capture groups → column names automatically
pattern = (
    r"service=(?P<service>\w+)\s+"
    r"version=(?P<version>\d+\.\d+\.\d+)\s+"
    r"id=(?P<id>[0-9a-f-]{36})"
)

parsed: pd.DataFrame = events.str.extract(pattern)
print(parsed.to_string())
print(f"\nDtype of 'version' column: {parsed['version'].dtype}")
# tip: parsed['version'] is still strings — see Example 9 for semver sorting

### Example 4 — Batch string templating / message generation
> **Why bother?**  Build thousands of SQL snippets, notification messages, or config lines from a template in microseconds — no `str.format` loop needed.

In [ ]:
"""Example 4: vectorised string formatting / templating."""

users = pd.DataFrame({
    "username": ["alice", "bob", "carol", "dave", "eve"],
    "role":     ["admin", "viewer", "editor", "viewer", "admin"],
    "quota_mb": [500, 100, 250, 100, 1000],
})

# ── approach A: arithmetic on string Series ──────────────────────────────────
# pandas overloads + for string concatenation — no loop, no f-string in Python
messages: pd.Series = (
    "GRANT " + users["role"].str.upper()
    + " TO " + users["username"]
    + " QUOTA " + users["quota_mb"].astype(str) + "MB;"
)
print("Generated SQL-like grants:")
print(messages.to_string(), "\n")

# ── approach B: DataFrame.apply with a format string (cleaner for complex templates)
TEMPLATE = "Hello {username}, your {role} account has {quota_mb} MB quota."

def render_row(row: pd.Series) -> str:
    """Render a user record into a notification string.

    Args:
        row: A single row from the users DataFrame with fields
             ``username``, ``role``, and ``quota_mb``.

    Returns:
        Formatted notification message string.
    """
    return TEMPLATE.format(**row)

notifications: pd.Series = users.apply(render_row, axis=1)
print("Notification messages:")
print(notifications.to_string())

### Example 5 — Multi-value string matching / contains any
> **Why bother?**  Quickly flag lines that match any of N patterns (error codes, banned words, PII markers) — the Pandas equivalent of `any(p in s for p in patterns)`, applied to every row at once.

In [ ]:
"""Example 5: multi-pattern matching — flag rows matching ANY of N patterns."""

descriptions: pd.Series = pd.Series([
    "NullPointerException in AuthService.login",
    "User alice logged in successfully",
    "Connection timeout to payment-db:5432",
    "OutOfMemoryError: Java heap space",
    "Request completed in 42ms",
    "FATAL: disk full on /var/data",
    "IndexOutOfBoundsException at pos 0",
    "Scheduled job finished",
])

CRITICAL_PATTERNS: list[str] = [
    r"Exception",
    r"Error",
    r"FATAL",
    r"timeout",
]

# Join patterns into a single alternation regex — one pass over the data
combined_re: str = "|".join(CRITICAL_PATTERNS)
is_critical: pd.Series = descriptions.str.contains(combined_re, case=False, regex=True)

print("Critical lines:")
print(descriptions[is_critical].to_string())
print(f"\n{is_critical.sum()} / {len(descriptions)} lines are critical")

#### ⏱ Speed check — string matching at scale

In [ ]:
BIG_LINES: list[str] = make_random_strings(100_000, max_len=120)
big_series = pd.Series(BIG_LINES)
big_arr    = np.array(BIG_LINES)

# Python loop
%timeit [any(p in s for p in ["abc", "xyz", "mno"]) for s in BIG_LINES]

# Pandas vectorised (single regex pass)
%timeit big_series.str.contains("abc|xyz|mno", regex=True)

---
## Part 2 — Log Processing

Logs are the lifeblood of backend engineering. Most SE tools parse them line-by-line in Python loops. NumPy/Pandas let you treat 1 000 000 log lines as a single data structure.

In [ ]:
"""Shared log-line fixture used by Examples 6-9."""

from datetime import datetime, timedelta

LOG_LEVELS = ["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"]
SERVICES   = ["auth", "payment", "inventory", "gateway", "worker"]
MESSAGES   = [
    "request completed",
    "cache miss",
    "retrying connection",
    "user not found",
    "timeout exceeded",
    "job dispatched",
    "rate limit hit",
    "disk usage high",
]

N_LOGS = 50_000

def make_log_lines(n: int) -> list[str]:
    """Generate synthetic nginx/app-style log lines.

    Args:
        n: Number of log lines to generate.

    Returns:
        List of log strings in the format::

            2024-01-15 08:32:01 [ERROR] payment: timeout exceeded duration_ms=342
    """
    base = datetime(2024, 1, 15, 0, 0, 0)
    lines: list[str] = []
    for i in range(n):
        ts   = base + timedelta(seconds=i * 2)
        lvl  = random.choices(LOG_LEVELS, weights=[20, 50, 15, 12, 3])[0]
        svc  = random.choice(SERVICES)
        msg  = random.choice(MESSAGES)
        dur  = random.randint(1, 9999)
        lines.append(
            f"{ts:%Y-%m-%d %H:%M:%S} [{lvl}] {svc}: {msg} duration_ms={dur}"
        )
    return lines

LOG_LINES: list[str] = make_log_lines(N_LOGS)
logs = pd.Series(LOG_LINES)
print(f"Generated {len(logs):,} log lines")
print(logs.iloc[0])
print(logs.iloc[-1])

### Example 6 — Parse all log fields into a structured DataFrame in one shot
> **Why bother?**  Once the log is a DataFrame you can filter, group, sort, and aggregate with SQL-like expressiveness — no custom parser class needed.

In [ ]:
"""Example 6: parse log lines → structured DataFrame."""

LOG_RE = (
    r"(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})"
    r" \[(?P<level>\w+)\]"
    r" (?P<service>\w+):"
    r" (?P<message>.+?)"
    r" duration_ms=(?P<duration_ms>\d+)"
)

# One regex call over the whole Series → DataFrame of named groups
df_logs: pd.DataFrame = logs.str.extract(LOG_RE)
df_logs["timestamp"]   = pd.to_datetime(df_logs["timestamp"])
df_logs["duration_ms"] = df_logs["duration_ms"].astype(int)

print(df_logs.dtypes, "\n")
print(df_logs.head(3).to_string())

# Immediately useful: aggregations
print("\n── Error count by service ──")
print(
    df_logs[df_logs["level"] == "ERROR"]
    .groupby("service")["level"]
    .count()
    .sort_values(ascending=False)
)

### Example 7 — Reformat / replace timestamps in bulk
> **Why bother?**  Log aggregators often require ISO-8601; your app emits `DD/MMM/YYYY`. Reformat 1 million lines without touching a for-loop.

In [ ]:
"""Example 7: bulk timestamp reformatting in log lines."""

# Simulate nginx-style timestamps: "15/Jan/2024:08:32:01 +0000"
def make_nginx_lines(n: int) -> list[str]:
    """Create synthetic nginx access log lines with nginx timestamp format.

    Args:
        n: Number of lines to generate.

    Returns:
        List of nginx-format log strings.
    """
    base = datetime(2024, 1, 15, 0, 0, 0)
    return [
        f'127.0.0.{i % 255} - - [{(base + timedelta(seconds=i)):%d/%b/%Y:%H:%M:%S +0000}]'
        f' "GET /api/v1/resource HTTP/1.1" {random.choice([200,404,500])} {random.randint(100,9999)}'
        for i in range(n)
    ]

NGINX_N = 100_000
nginx_lines: list[str] = make_nginx_lines(NGINX_N)
nginx_series = pd.Series(nginx_lines)

# ── Python loop baseline ─────────────────────────────────────────────────────
NGINX_TS_RE = re.compile(r"\[(\d{2}/\w{3}/\d{4}:\d{2}:\d{2}:\d{2} \+\d{4})\]")

def reformat_loop(lines: list[str]) -> list[str]:
    """Reformat nginx timestamps to ISO-8601 using a Python loop.

    Args:
        lines: List of raw nginx log line strings.

    Returns:
        New list with timestamps replaced by ISO-8601 strings.
    """
    out: list[str] = []
    for line in lines:
        m = NGINX_TS_RE.search(line)
        if m:
            ts = datetime.strptime(m.group(1), "%d/%b/%Y:%H:%M:%S %z")
            out.append(line.replace(m.group(0), f"[{ts.isoformat()}]"))
        else:
            out.append(line)
    return out

# ── Pandas vectorised approach ───────────────────────────────────────────────
TS_PATTERN = r"\[(\d{2}/\w{3}/\d{4}:\d{2}:\d{2}:\d{2} \+\d{4})\]"

def reformat_vectorised(series: pd.Series) -> pd.Series:
    """Reformat nginx timestamps to ISO-8601 using vectorised Pandas ops.

    Args:
        series: Series of raw nginx log line strings.

    Returns:
        Series with timestamps replaced by ISO-8601 strings.
    """
    raw_ts: pd.Series = series.str.extract(TS_PATTERN, expand=False)
    iso_ts: pd.Series = pd.to_datetime(raw_ts, format="%d/%b/%Y:%H:%M:%S %z").dt.isoformat()
    # Replace the bracketed nginx timestamp with ISO form
    return series.str.replace(TS_PATTERN, lambda m: f"[{iso_ts[m.string[:10]]}]", regex=True)

# -- timeit comparison --
print("Reformatting", NGINX_N, "nginx log timestamps:\n")
%timeit reformat_loop(nginx_lines)

# Show a before/after sample
sample_out = reformat_loop(nginx_lines[:2])
print("\nBefore:", nginx_lines[0])
print("After: ", sample_out[0])

### Example 8 — Pivot log levels into a frequency table (groupby + unstack)
> **Why bother?**  "How many ERRORs per service per hour?" is a two-liner with Pandas. In plain Python it's a `defaultdict(lambda: defaultdict(int))` and a nested loop.

In [ ]:
"""Example 8: log-level frequency pivot table."""
from collections import defaultdict

# Re-use df_logs built in Example 6
df_logs["hour"] = df_logs["timestamp"].dt.floor("h")

# ── Pandas: two lines ────────────────────────────────────────────────────────
pivot: pd.DataFrame = (
    df_logs.groupby(["hour", "level"])
           .size()
           .unstack(fill_value=0)
)
print("── Errors/service/hour (first 5 hours, subset of levels) ──")
print(pivot[["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"]].head().to_string())

# ── Python equivalent (for contrast) ─────────────────────────────────────────
def pivot_loop(rows: list[tuple[Any, Any]]) -> dict[Any, dict[Any, int]]:
    """Build an hour×level count table using plain Python dicts.

    Args:
        rows: List of (hour, level) tuples.

    Returns:
        Nested dict mapping hour → level → count.
    """
    counts: dict[Any, dict[Any, int]] = defaultdict(lambda: defaultdict(int))
    for hour, level in rows:
        counts[hour][level] += 1
    return counts

rows: list[tuple] = list(zip(df_logs["hour"], df_logs["level"]))

print("\n── timeit: groupby+unstack vs. nested defaultdict ──")
%timeit df_logs.groupby(["hour", "level"]).size().unstack(fill_value=0)
%timeit pivot_loop(rows)

### Example 9 — Rolling SLA: p95 latency over a sliding window
> **Why bother?**  Alerting on "p95 latency exceeded 500 ms in the last 5 minutes" is a window aggregation. In NumPy/Pandas it's `rolling(300).quantile(0.95)` — no deque, no heap.

In [ ]:
"""Example 9: rolling p95 latency — sliding-window SLA monitoring."""
import collections

# Use the parsed duration_ms from df_logs
ts_latency = df_logs.set_index("timestamp")["duration_ms"].sort_index()

# 5-minute rolling p95
WINDOW = "5min"
SLA_MS = 500

rolling_p95: pd.Series = ts_latency.rolling(WINDOW).quantile(0.95)
breaches: pd.Series    = rolling_p95[rolling_p95 > SLA_MS]

print(f"Total time-points: {len(rolling_p95):,}")
print(f"SLA breaches (p95 > {SLA_MS} ms): {len(breaches):,}")
print(f"\nWorst 5 windows:")
print(rolling_p95.nlargest(5).to_string())

# ── Python loop equivalent for contrast ──────────────────────────────────────
def rolling_p95_loop(durations: list[int], window: int = 300) -> list[float]:
    """Compute rolling p95 using a deque, Python-only.

    Args:
        durations: List of latency values in ms.
        window: Window size in number of samples.

    Returns:
        List of p95 values (NaN represented as -1 before window fills).
    """
    dq: collections.deque[int] = collections.deque(maxlen=window)
    result: list[float] = []
    for v in durations:
        dq.append(v)
        if len(dq) == window:
            result.append(float(np.percentile(list(dq), 95)))
        else:
            result.append(float("nan"))
    return result

vals = ts_latency.tolist()
print("\n── timeit: rolling().quantile() vs. Python deque loop ──")
%timeit ts_latency.rolling(300).quantile(0.95)
%timeit rolling_p95_loop(vals, window=300)

---
## Part 3 — Data Transformation, Lookups & Conditional Logic

These are the patterns you reach for when you have an ETL step, a config hydration pass, or any "enrich this record" operation.

### Example 10 — O(1) bulk value mapping with `Series.map`
> **Why bother?**  Translating status codes → descriptions, user IDs → names, or country codes → regions is a dictionary lookup on every row. `Series.map` does it in a single C-speed pass.

In [ ]:
"""Example 10: bulk dictionary lookups with Series.map."""

HTTP_STATUS: dict[int, str] = {
    200: "OK", 201: "Created", 204: "No Content",
    301: "Moved Permanently", 302: "Found",
    400: "Bad Request", 401: "Unauthorized", 403: "Forbidden",
    404: "Not Found", 422: "Unprocessable Entity",
    429: "Too Many Requests", 500: "Internal Server Error",
    502: "Bad Gateway", 503: "Service Unavailable",
}

N_REQUESTS = 500_000
status_codes: np.ndarray = RNG.choice(list(HTTP_STATUS.keys()), size=N_REQUESTS)
codes_series = pd.Series(status_codes)

# ── Pandas map ────────────────────────────────────────────────────────────────
descriptions: pd.Series = codes_series.map(HTTP_STATUS)

print("Sample:")
print(pd.DataFrame({"code": codes_series, "description": descriptions}).head(8).to_string())
print(f"\nUnknown codes (NaN): {descriptions.isna().sum()}")

# ── timeit ────────────────────────────────────────────────────────────────────
codes_list: list[int] = status_codes.tolist()
print("\n── timeit: Series.map vs. list comprehension lookup ──")
%timeit codes_series.map(HTTP_STATUS)
%timeit [HTTP_STATUS.get(c, "Unknown") for c in codes_list]

### Example 11 — `np.where` / `np.select` as a vectorised if-elif-else
> **Why bother?**  Classifying records into categories (severity tiers, billing bands, SLA buckets) without a loop or a chain of `apply` calls.

In [ ]:
"""Example 11: np.where / np.select — vectorised conditional classification."""

latencies: np.ndarray = RNG.integers(1, 5000, size=200_000)

# ── np.where: single binary condition (like a ternary) ───────────────────────
sla_ok: np.ndarray = np.where(latencies < 200, "within_sla", "breached_sla")

# ── np.select: multi-branch elif chain ───────────────────────────────────────
conditions: list[np.ndarray] = [
    latencies < 100,
    latencies < 500,
    latencies < 2000,
]
labels: list[str] = ["fast", "normal", "slow"]
tier: np.ndarray = np.select(conditions, labels, default="critical")

unique, counts = np.unique(tier, return_counts=True)
print("Latency tier distribution:")
for t, c in sorted(zip(unique, counts)):
    print(f"  {t:10s}: {c:7,}  ({c/len(tier)*100:.1f}%)")

# ── timeit ────────────────────────────────────────────────────────────────────
lat_list: list[int] = latencies.tolist()

def classify_loop(lats: list[int]) -> list[str]:
    """Classify latencies into tiers using a Python loop.

    Args:
        lats: List of latency values in milliseconds.

    Returns:
        List of tier strings: 'fast', 'normal', 'slow', or 'critical'.
    """
    result: list[str] = []
    for v in lats:
        if v < 100:
            result.append("fast")
        elif v < 500:
            result.append("normal")
        elif v < 2000:
            result.append("slow")
        else:
            result.append("critical")
    return result

print("\n── timeit: np.select vs. Python loop ──")
%timeit np.select(conditions, labels, default="critical")
%timeit classify_loop(lat_list)

### Example 12 — Fast deduplication and set operations on arrays
> **Why bother?**  Finding new feature flags added since the last deploy, removed users, or the intersection of two permission sets — NumPy has `np.unique`, `np.intersect1d`, `np.setdiff1d`, `np.union1d`.

In [ ]:
"""Example 12: deduplication and set operations with NumPy."""

# Simulate two deploys' feature flag lists
N_FLAGS = 300_000
all_flags = np.array([f"flag_{i:05d}" for i in range(N_FLAGS)])

prev_deploy: np.ndarray = RNG.choice(all_flags, size=200_000, replace=False)
curr_deploy: np.ndarray = RNG.choice(all_flags, size=210_000, replace=False)

# ── deduplication ─────────────────────────────────────────────────────────────
unique_curr: np.ndarray = np.unique(curr_deploy)          # sorted, deduped

# ── set operations ────────────────────────────────────────────────────────────
added:   np.ndarray = np.setdiff1d(curr_deploy, prev_deploy)   # new in curr
removed: np.ndarray = np.setdiff1d(prev_deploy, curr_deploy)   # gone from prev
common:  np.ndarray = np.intersect1d(prev_deploy, curr_deploy)

print(f"Prev deploy flags : {len(np.unique(prev_deploy)):,}")
print(f"Curr deploy flags : {len(unique_curr):,}")
print(f"  Added   : {len(added):,}")
print(f"  Removed : {len(removed):,}")
print(f"  Common  : {len(common):,}")
print(f"\nSample added  : {added[:5]}")
print(f"Sample removed: {removed[:5]}")

# ── timeit: np.setdiff1d vs. Python set difference ───────────────────────────
prev_set = set(prev_deploy.tolist())
curr_set = set(curr_deploy.tolist())

print("\n── timeit: set difference, NumPy vs. Python set ──")
%timeit np.setdiff1d(curr_deploy, prev_deploy)
%timeit curr_set - prev_set

### Example 13 — Batch JSON field extraction
> **Why bother?**  Kafka consumers, webhook handlers, and audit-log pipelines deal with thousands of JSON blobs per second. Pandas can turn a column of JSON strings into typed columns in one step.

In [ ]:
"""Example 13: bulk JSON field extraction with pd.json_normalize."""

ACTIONS   = ["login", "logout", "purchase", "view", "click", "search"]
PLATFORMS = ["web", "ios", "android"]

def make_event_json(n: int) -> list[str]:
    """Generate synthetic analytics event JSON strings.

    Args:
        n: Number of JSON event strings to generate.

    Returns:
        List of JSON strings with fields: user_id, action, platform,
        metadata.duration_ms, metadata.retries.
    """
    events: list[str] = []
    for i in range(n):
        payload = {
            "user_id":  f"u{RNG.integers(1, 10_000):05d}",
            "action":   random.choice(ACTIONS),
            "platform": random.choice(PLATFORMS),
            "metadata": {
                "duration_ms": int(RNG.integers(10, 3000)),
                "retries":     int(RNG.integers(0, 4)),
            },
        }
        events.append(json.dumps(payload))
    return events

N_EVENTS = 20_000
event_jsons: list[str] = make_event_json(N_EVENTS)

# ── loop baseline ─────────────────────────────────────────────────────────────
def extract_loop(blobs: list[str]) -> list[dict[str, Any]]:
    """Parse JSON blobs and extract fields using a Python loop.

    Args:
        blobs: List of raw JSON strings.

    Returns:
        List of flat dicts with keys: user_id, action, platform,
        duration_ms, retries.
    """
    records: list[dict[str, Any]] = []
    for blob in blobs:
        d = json.loads(blob)
        records.append({
            "user_id":     d["user_id"],
            "action":      d["action"],
            "platform":    d["platform"],
            "duration_ms": d["metadata"]["duration_ms"],
            "retries":     d["metadata"]["retries"],
        })
    return records

# ── Pandas approach ────────────────────────────────────────────────────────────
parsed_dicts: list[dict] = [json.loads(b) for b in event_jsons]
df_events: pd.DataFrame  = pd.json_normalize(parsed_dicts)   # flattens nested keys
# metadata.duration_ms, metadata.retries become column names automatically

print(df_events.dtypes, "\n")
print(df_events.head(4).to_string())
print("\n── Action distribution ──")
print(df_events["action"].value_counts().to_string())

print("\n── timeit: json_normalize vs. loop extraction ──")
%timeit pd.json_normalize([json.loads(b) for b in event_jsons])
%timeit extract_loop(event_jsons)

### Example 14 — Batch file path manipulation
> **Why bother?**  Build tooling, release pipelines, and test scaffolding all deal with lists of file paths. Pandas string ops replace `os.path` loops elegantly.

In [ ]:
"""Example 14: vectorised file path analysis with Pandas str API."""
import os

EXTENSIONS = [".py", ".go", ".ts", ".rs", ".yaml", ".json", ".md", ".sql"]
DIRS = [
    "/srv/app/src", "/srv/app/tests", "/srv/infra/k8s",
    "/srv/infra/terraform", "/srv/docs", "/srv/migrations",
]

def make_file_paths(n: int) -> list[str]:
    """Generate synthetic repository-style file paths.

    Args:
        n: Number of paths to generate.

    Returns:
        List of absolute path strings with mixed extensions and depths.
    """
    paths: list[str] = []
    for _ in range(n):
        depth = random.randint(1, 4)
        parts = [random.choice(DIRS)]
        for _ in range(depth):
            parts.append(
                "".join(random.choices(string.ascii_lowercase, k=random.randint(4, 12)))
            )
        ext = random.choice(EXTENSIONS)
        paths.append(os.path.join(*parts) + ext)
    return paths

N_PATHS = 100_000
file_paths: list[str] = make_file_paths(N_PATHS)
ps = pd.Series(file_paths)

# ── vectorised path analysis ──────────────────────────────────────────────────
extensions: pd.Series = ps.str.extract(r"(\.[^./]+)$", expand=False)
basenames:  pd.Series = ps.str.extract(r"/([^/]+)$",   expand=False)
is_test:    pd.Series = ps.str.contains("/tests/")

print("Extension breakdown:")
print(extensions.value_counts().to_string())
print(f"\nTest files: {is_test.sum():,} / {N_PATHS:,}")
print(f"\nSample basenames:\n{basenames.head(5).to_string()}")

# ── Rename: swap extension across the whole corpus ───────────────────────────
# e.g. migrate from .yaml to .yml for all infra files
infra_mask: pd.Series  = ps.str.startswith("/srv/infra/")
renamed:    pd.Series  = ps.copy()
renamed[infra_mask] = ps[infra_mask].str.replace(r"\.yaml$", ".yml", regex=True)
print(f"\nRenamed {(renamed != ps).sum():,} paths (.yaml → .yml in /srv/infra/)")

---
## Part 4 — Performance, Sorting & Numerical Engineering Utilities

The final set covers patterns that appear in CI scripts, metrics pipelines, config validation, and release tooling.

### Example 15 — Top-N and argsort (leaderboards, hot-paths, heaviest files)
> **Why bother?**  Find the N slowest endpoints, the N largest files, or the N most-erroring users. `np.argpartition` is O(n) — faster than sorting the whole array.

In [ ]:
"""Example 15: Top-N via np.argpartition (O(n)) vs. sorted (O(n log n))."""

N_ENDPOINTS = 1_000_000
endpoint_ids: np.ndarray  = np.array([f"ep_{i:06d}" for i in range(N_ENDPOINTS)])
endpoint_p99: np.ndarray  = RNG.exponential(scale=200, size=N_ENDPOINTS).astype(int)

TOP_N = 20

# ── O(n) partial sort via argpartition ────────────────────────────────────────
def top_n_numpy(values: np.ndarray, n: int) -> np.ndarray:
    """Return the indices of the top-N largest values using O(n) partition.

    Args:
        values: Array of numeric values to rank.
        n: Number of top elements to return.

    Returns:
        Array of indices into ``values``, sorted from highest to lowest.
    """
    # argpartition guarantees top-N are in the last n slots — then sort just those
    part: np.ndarray = np.argpartition(values, -n)[-n:]
    return part[np.argsort(values[part])[::-1]]

# ── Python baseline ───────────────────────────────────────────────────────────
def top_n_loop(values: list[int], n: int) -> list[int]:
    """Return indices of the top-N largest values using Python sorted().

    Args:
        values: List of numeric values to rank.
        n: Number of top elements to return.

    Returns:
        List of indices into ``values``, sorted from highest to lowest.
    """
    return sorted(range(len(values)), key=lambda i: values[i], reverse=True)[:n]

top_idx = top_n_numpy(endpoint_p99, TOP_N)
print(f"Top {TOP_N} slowest endpoints (p99 ms):")
for rank, idx in enumerate(top_idx, 1):
    print(f"  {rank:2d}. {endpoint_ids[idx]}  {endpoint_p99[idx]:6,} ms")

p99_list: list[int] = endpoint_p99.tolist()
print(f"\n── timeit: argpartition O(n) vs. sorted O(n log n) — {N_ENDPOINTS:,} endpoints ──")
%timeit top_n_numpy(endpoint_p99, TOP_N)
%timeit top_n_loop(p99_list, TOP_N)

### Example 16 — Numerical normalisation: clip, scale, percentile-rank
> **Why bother?**  Rate-limiting, scoring, and alerting all need values normalised, clipped, or ranked. NumPy does it without a single explicit loop.

In [ ]:
"""Example 16: clip, min-max scale, and percentile rank in NumPy/Pandas."""

raw_scores: np.ndarray = RNG.normal(loc=500, scale=200, size=100_000).astype(int)

# ── clip outliers ─────────────────────────────────────────────────────────────
clipped: np.ndarray = np.clip(raw_scores, a_min=0, a_max=1000)

# ── min-max normalise to [0, 1] ───────────────────────────────────────────────
lo, hi = clipped.min(), clipped.max()
normalised: np.ndarray = (clipped - lo) / (hi - lo)

# ── percentile rank (0-100) ───────────────────────────────────────────────────
# pd.Series.rank with pct=True returns fractional rank
pct_rank: pd.Series = pd.Series(raw_scores).rank(pct=True) * 100

print(f"Raw    — min:{raw_scores.min():5d}  max:{raw_scores.max():5d}  mean:{raw_scores.mean():.1f}")
print(f"Clipped— min:{clipped.min():5d}  max:{clipped.max():5d}")
print(f"Normed — min:{normalised.min():.3f}  max:{normalised.max():.3f}  mean:{normalised.mean():.3f}")
print(f"PctRank— sample (first 5): {pct_rank.head().round(1).tolist()}")

# ── timeit: numpy clip vs. Python list comprehension ──────────────────────────
raw_list: list[int] = raw_scores.tolist()
print("\n── timeit: np.clip vs. list comprehension clamp ──")
%timeit np.clip(raw_scores, 0, 1000)
%timeit [max(0, min(1000, v)) for v in raw_list]

### Example 17 — Diff / change-detection between two config snapshots
> **Why bother?**  During a config audit you want to know which keys changed value between two YAML loads. `pd.DataFrame.compare` gives you a structured diff in one call.

In [ ]:
"""Example 17: structured config diff with pd.DataFrame.compare."""

# Simulate two snapshots of a microservice config table
config_keys = [
    "max_connections", "timeout_ms", "retry_count", "log_level",
    "cache_ttl_s", "rate_limit_rps", "feature_new_ui", "feature_dark_mode",
    "db_pool_size", "worker_threads",
]

before = pd.DataFrame({
    "key":   config_keys,
    "value": ["100", "500", "3", "INFO", "300", "1000",
              "false", "false", "10", "4"],
})

after = pd.DataFrame({
    "key":   config_keys,
    "value": ["150", "500", "5", "DEBUG", "300", "1000",
              "true", "false", "10", "8"],   # max_conn, retry, log_level, new_ui, workers changed
})

before_indexed = before.set_index("key")
after_indexed  = after.set_index("key")

# ── pd.DataFrame.compare ─────────────────────────────────────────────────────
diff: pd.DataFrame = before_indexed.compare(after_indexed, result_names=("before", "after"))

if diff.empty:
    print("No changes detected.")
else:
    print("Changed config keys:")
    print(diff.to_string())
    print(f"\n{len(diff)} key(s) changed out of {len(config_keys)}")

# ── programmatic change list ──────────────────────────────────────────────────
changed_keys: list[str] = diff.index.tolist()
print(f"\nKeys to audit: {changed_keys}")

### Example 18 — Vectorised IP address validation and CIDR range check
> **Why bother?**  Firewall rule auditing, access-log analysis, and network config tooling all need to classify thousands of IPs. NumPy integer arithmetic is dramatically faster than `ipaddress` in a loop.

In [ ]:
"""Example 18: vectorised IP range classification with NumPy integer arithmetic."""
import ipaddress
import struct

# ── helper: IP string → uint32 (vectorised via np.vectorize) ─────────────────
def ip_to_int(ip: str) -> int:
    """Convert a dotted-decimal IPv4 string to its 32-bit integer value.

    Args:
        ip: IPv4 address string, e.g. ``"192.168.1.1"``.

    Returns:
        Unsigned 32-bit integer representation of the address.

    Raises:
        ValueError: If ``ip`` is not a valid IPv4 address string.
    """
    return struct.unpack("!I", ipaddress.IPv4Address(ip).packed)[0]

vec_ip_to_int = np.vectorize(ip_to_int, otypes=[np.uint32])

# ── CIDR range bounds ────────────────────────────────────────────────────────
def cidr_bounds(cidr: str) -> tuple[int, int]:
    """Return the (network_address, broadcast_address) integers for a CIDR.

    Args:
        cidr: CIDR notation string, e.g. ``"10.0.0.0/8"``.

    Returns:
        Tuple of (first_ip_int, last_ip_int) inclusive range.
    """
    net = ipaddress.IPv4Network(cidr, strict=False)
    return int(net.network_address), int(net.broadcast_address)

PRIVATE_CIDRS: list[str] = ["10.0.0.0/8", "172.16.0.0/12", "192.168.0.0/16"]
cidr_ranges: list[tuple[int, int]] = [cidr_bounds(c) for c in PRIVATE_CIDRS]

# ── synthetic IP dataset ──────────────────────────────────────────────────────
def make_ips(n: int) -> list[str]:
    """Generate a mix of private and public IPv4 addresses.

    Args:
        n: Total number of IP strings to generate.

    Returns:
        List of dotted-decimal IPv4 address strings.
    """
    ips: list[str] = []
    for _ in range(n):
        if random.random() < 0.4:   # 40% private
            ips.append(f"192.168.{random.randint(0,255)}.{random.randint(0,255)}")
        else:
            ips.append(
                f"{random.randint(1,223)}.{random.randint(0,255)}."
                f"{random.randint(0,255)}.{random.randint(0,255)}"
            )
    return ips

N_IPS = 50_000
ip_strings: list[str] = make_ips(N_IPS)
ip_arr     = np.array(ip_strings)

# ── vectorised classification ─────────────────────────────────────────────────
ip_ints: np.ndarray = vec_ip_to_int(ip_arr)

# Build a boolean mask: is ANY of the private CIDRs satisfied?
is_private: np.ndarray = np.zeros(N_IPS, dtype=bool)
for lo, hi in cidr_ranges:
    is_private |= (ip_ints >= lo) & (ip_ints <= hi)

print(f"Total IPs   : {N_IPS:,}")
print(f"Private IPs : {is_private.sum():,}  ({is_private.mean()*100:.1f}%)")
print(f"Public IPs  : {(~is_private).sum():,}")
print(f"\nFirst 5 private: {ip_arr[is_private][:5].tolist()}")

# ── timeit ────────────────────────────────────────────────────────────────────
def classify_loop_ip(ips: list[str]) -> list[bool]:
    """Classify IPs as private using Python ipaddress module loop.

    Args:
        ips: List of dotted-decimal IPv4 address strings.

    Returns:
        List of booleans, True if the IP is in a private range.
    """
    private_nets = [ipaddress.IPv4Network(c) for c in PRIVATE_CIDRS]
    result: list[bool] = []
    for ip in ips:
        addr = ipaddress.IPv4Address(ip)
        result.append(any(addr in net for net in private_nets))
    return result

print(f"\n── timeit: NumPy integer range check vs. ipaddress loop — {N_IPS:,} IPs ──")
%timeit vec_ip_to_int(ip_arr); [np.zeros(N_IPS, dtype=bool) | ((ip_ints >= lo) & (ip_ints <= hi)) for lo, hi in cidr_ranges]
%timeit classify_loop_ip(ip_strings)

---
## Cheat-Sheet Summary

| Task | NumPy idiom | Pandas idiom |
|---|---|---|
| Length of N strings | `np.char.str_len(arr)` | `s.str.len()` |
| Normalise / strip / lower | `np.char.lower(arr)` | `s.str.strip().str.lower()` |
| Regex extract to columns | — | `s.str.extract(r"(?P<a>...)")` |
| Multi-pattern flag | `np.char.find` | `s.str.contains("a\|b\|c")` |
| Batch template render | `np.vectorize(fn)` | `df.apply(render, axis=1)` |
| Parse structured log | — | `s.str.extract(LOG_RE)` |
| Reformat timestamps | `pd.to_datetime` + `.dt.strftime` | same |
| Pivot count table | — | `groupby().size().unstack()` |
| Rolling window stat | — | `rolling(w).quantile(0.95)` |
| Dictionary lookup | — | `s.map(d)` |
| if-elif-else on array | `np.select(conds, vals)` | `pd.cut` / `np.select` |
| Deduplication | `np.unique(arr)` | `s.drop_duplicates()` |
| Set diff / intersect | `np.setdiff1d / np.intersect1d` | `pd.Index.difference` |
| Flat JSON → DataFrame | — | `pd.json_normalize(dicts)` |
| Path / extension ops | — | `s.str.extract(r"(\.\w+)$")` |
| Top-N (O(n)) | `np.argpartition(arr, -N)[-N:]` | `s.nlargest(N)` |
| Clip + normalise | `np.clip(arr, lo, hi)` | `s.clip(lo, hi)` |
| Config diff | — | `df.compare(df2)` |
| IP range classification | `np.vectorize(ip_to_int)` + masks | — |

> **Rule of thumb:** if you're writing a `for` loop over a list and the loop body is a pure transformation (no I/O, no external state), NumPy/Pandas will be faster **and** shorter.